In [ ]:
'''Exercise 1 - Pandas: Advanced groupby and agg

Load enriched_data.csv from the Day 9
hospital project (or recreate it).

1. Using groupby on specialty:
   Show count, total fee, avg fee, max fee.
   Use .agg({'fee': ['count','sum','mean','max']})
   Rename columns cleanly after aggregating.

2. Using groupby on age_group:
   Show count of patients and avg fee.
   Sort by avg fee descending.

3. Using groupby on day_of_week:
   Show total fee per day.
   Which day generated the most revenue?

4. Using groupby on fee_category:
   Show count and avg patient age.
   Is there a pattern?

5. Cross-tabulation:
   Count how many appointments each
   specialty handled per fee_category.
   Use pd.crosstab() or groupby on two columns.

Concepts: groupby().agg(), rename(),
          crosstab(), sort_values(), idxmax()
'''

import pandas as pd

df = pd.read_csv("enriched_data.csv")

# Group by specialty
specialty_stats = df.groupby("specialty").agg({
    "fee": ["count", "sum", "mean", "max"]
})

# Clean column names
specialty_stats.columns = ["count", "total_fee", "avg_fee", "max_fee"]
print(specialty_stats)




                  count  total_fee     avg_fee  max_fee
specialty                                              
Cardiology            2       1400  700.000000      900
Dermatology           1        300  300.000000      300
General Medicine      3        550  183.333333      200
Neurology             2       1450  725.000000      850
Orthopedics           2       1150  575.000000      750


In [3]:
age_stats = df.groupby("age_group").agg({
    "patient_name": "count",
    "fee": "mean"
})

age_stats = age_stats.rename(columns={
    "patient_name": "patient_count",
    "fee": "avg_fee"
})

# Sort by avg fee descending
age_stats = age_stats.sort_values("avg_fee", ascending=False)
print(age_stats)

           patient_count     avg_fee
age_group                           
Senior                 1  900.000000
Adult                  6  483.333333
Young                  3  350.000000


In [4]:
day_stats = df.groupby("day_of_week")["fee"].sum()
print(day_stats)

# Which day generated the most revenue?
print("Top revenue day:", day_stats.idxmax())

day_of_week
Friday       1600
Monday        200
Saturday      600
Sunday        300
Thursday      350
Tuesday       900
Wednesday     900
Name: fee, dtype: int64
Top revenue day: Friday


In [6]:
print(df.columns)

Index(['appt_id', 'patient_name', 'age_group', 'patient_city', 'doctor_name',
       'specialty', 'seniority', 'appt_date', 'day_of_week', 'diagnosis',
       'fee', 'fee_category'],
      dtype='str')


In [10]:
fee_stats = df.groupby(["fee_category", "age_group"]).size().unstack(fill_value=0)
print(fee_stats)

age_group     Adult  Senior  Young
fee_category                      
High              2       1      0
Low               2       0      2
Medium            2       0      1


In [11]:
crosstab = pd.crosstab(df["specialty"], df["fee_category"])
print(crosstab)

fee_category      High  Low  Medium
specialty                          
Cardiology           1    0       1
Dermatology          0    1       0
General Medicine     0    3       0
Neurology            1    0       1
Orthopedics          1    0       1


In [ ]:
'''Exercise 2 - Nested JSON handling

Create a file called hospital_report.json
with this structure:

{
  "report_date": "2024-01-20",
  "hospital": "City General",
  "departments": [
    {
      "name": "Cardiology",
      "doctor": "Dr. Anil Mehta",
      "appointments": [
        {"appt_id": 1, "patient": "Raj Kumar",
         "fee": 500, "date": "2024-01-10"},
        {"appt_id": 7, "patient": "Ana Gonzalez",
         "fee": 900, "date": "2024-01-16"}
      ]
    },
    {
      "name": "General Medicine",
      "doctor": "Dr. Carlos Torres",
      "appointments": [
        {"appt_id": 2, "patient": "Sneha Patel",
         "fee": 200, "date": "2024-01-11"},
        {"appt_id": 6, "patient": "Raj Kumar",
         "fee": 200, "date": "2024-01-15"},
        {"appt_id": 9, "patient": "Emily Carter",
         "fee": 150, "date": "2024-01-18"}
      ]
    }
  ]
}

Write code to:
1. Load the JSON file.
2. Print report_date and hospital name.
3. Loop through each department and print:
   - Department name
   - Doctor name
   - Number of appointments
   - Total fee collected
4. Flatten the nested structure into a
   pandas DataFrame with columns:
   department, doctor, appt_id,
   patient, fee, date.
   Do this without using json_normalize
   — use a loop and list of dicts.
5. Find which department earned more.
6. Export flattened DataFrame to
   department_report.csv.

Concepts: nested json, loop, list of dicts,
          pd.DataFrame(), to_csv()


---------------------------------------------''' 
import json

hospital_report = {
    "report_date": "2024-01-20",
    "hospital": "City General",
    "departments": [
        {
            "name": "Cardiology",
            "doctor": "Dr. Anil Mehta",
            "appointments": [
                {"appt_id": 1, "patient": "Raj Kumar", "fee": 500, "date": "2024-01-10"},
                {"appt_id": 7, "patient": "Ana Gonzalez", "fee": 900, "date": "2024-01-16"}
            ]
        },
        {
            "name": "General Medicine",
            "doctor": "Dr. Carlos Torres",
            "appointments": [
                {"appt_id": 2, "patient": "Sneha Patel", "fee": 200, "date": "2024-01-11"},
                {"appt_id": 6, "patient": "Raj Kumar", "fee": 200, "date": "2024-01-15"},
                {"appt_id": 9, "patient": "Emily Carter", "fee": 150, "date": "2024-01-18"}
            ]
        }
    ]
}

with open("hospital_report.json", "w") as f:
    json.dump(hospital_report, f, indent=2)

print("hospital_report.json created successfully!")


















hospital_report.json created successfully!


In [14]:

import pandas as pd

def process_report():
    # 1. Load the JSON file
    with open("hospital_report.json", "r") as f:
        data = json.load(f)

    # 2. Print report_date and hospital name
    print("Report Date:", data["report_date"])
    print("Hospital:", data["hospital"])

    # 3. Loop through each department
    for dept in data["departments"]:
        dept_name = dept["name"]
        doctor = dept["doctor"]
        appointments = dept["appointments"]

        num_appts = len(appointments)
        total_fee = sum(appt["fee"] for appt in appointments)

        print(f"\nDepartment: {dept_name}")
        print(f"Doctor: {doctor}")
        print(f"Appointments: {num_appts}")
        print(f"Total Fee: {total_fee}")

    # 4. Flatten into DataFrame
    rows = []
    for dept in data["departments"]:
        for appt in dept["appointments"]:
            rows.append({
                "department": dept["name"],
                "doctor": dept["doctor"],
                "appt_id": appt["appt_id"],
                "patient": appt["patient"],
                "fee": appt["fee"],
                "date": appt["date"]
            })

    df = pd.DataFrame(rows)
    print("\nFlattened DataFrame:")
    print(df.head())

    # 5. Find which department earned more
    dept_revenue = df.groupby("department")["fee"].sum()
    top_dept = dept_revenue.idxmax()
    print("\nRevenue by Department:")
    print(dept_revenue)
    print("Top earning department:", top_dept)

    # 6. Export to CSV
    df.to_csv("department_report.csv", index=False)

if __name__ == "__main__":
    process_report()

Report Date: 2024-01-20
Hospital: City General

Department: Cardiology
Doctor: Dr. Anil Mehta
Appointments: 2
Total Fee: 1400

Department: General Medicine
Doctor: Dr. Carlos Torres
Appointments: 3
Total Fee: 550

Flattened DataFrame:
         department             doctor  appt_id       patient  fee        date
0        Cardiology     Dr. Anil Mehta        1     Raj Kumar  500  2024-01-10
1        Cardiology     Dr. Anil Mehta        7  Ana Gonzalez  900  2024-01-16
2  General Medicine  Dr. Carlos Torres        2   Sneha Patel  200  2024-01-11
3  General Medicine  Dr. Carlos Torres        6     Raj Kumar  200  2024-01-15
4  General Medicine  Dr. Carlos Torres        9  Emily Carter  150  2024-01-18

Revenue by Department:
department
Cardiology          1400
General Medicine     550
Name: fee, dtype: int64
Top earning department: Cardiology


In [15]:
'''Exercise 3 - Exception handling with
             file and data operations

Write a function called safe_load_csv(filepath)
that:
- Tries to load the file using pandas
- Handles FileNotFoundError with a clear message
- Handles pd.errors.EmptyDataError
- Handles any other exception with a generic
  error message
- Returns the DataFrame on success,
  None on failure

Write a function called safe_process(df, column)
that:
- Tries to compute mean of the given column
- Handles KeyError if column does not exist
- Handles TypeError if column is not numeric
- Returns the mean or None

Test with:
1. A valid file and valid column.
2. A filename that does not exist.
3. A valid file but wrong column name.
4. A valid file but a text column.

Concepts: try/except, FileNotFoundError,
          KeyError, TypeError,
          pd.errors.EmptyDataError'''

import pandas as pd

# 1. Safe CSV loader
def safe_load_csv(filepath):
    try:
        df = pd.read_csv(filepath)
        print(f"Loaded file successfully: {filepath}")
        return df
    except FileNotFoundError:
        print(f"Error: File not found → {filepath}")
        return None
    except pd.errors.EmptyDataError:
        print(f"Error: File is empty → {filepath}")
        return None
    except Exception as e:
        print(f"Unexpected error while loading {filepath}: {e}")
        return None


# 2. Safe processor
def safe_process(df, column):
    try:
        mean_val = df[column].mean()
        print(f"Mean of {column}: {mean_val}")
        return mean_val
    except KeyError:
        print(f"Error: Column '{column}' does not exist in DataFrame")
        return None
    except TypeError:
        print(f"Error: Column '{column}' is not numeric")
        return None
    except Exception as e:
        print(f"Unexpected error while processing column '{column}': {e}")
        return None


# 3. Testing
if __name__ == "__main__":
    # Case 1: Valid file and valid column
    df1 = safe_load_csv("processed/enriched_data.csv")
    if df1 is not None:
        safe_process(df1, "fee")   # numeric column

    # Case 2: File does not exist
    df2 = safe_load_csv("missing_file.csv")

    # Case 3: Valid file but wrong column name
    if df1 is not None:
        safe_process(df1, "wrong_column")

    # Case 4: Valid file but text column
    if df1 is not None:
        safe_process(df1, "patient_name")  # text column

Error: File not found → processed/enriched_data.csv
Error: File not found → missing_file.csv


In [ ]:
'''Exercise 4 - Modular code structure

Refactor the following into a clean module.

Create a file called data_utils.py with
these functions:

1. load_csv(filepath) — loads and returns df
2. clean_strings(df) — strips all string cols
3. add_fee_category(df, col='fee') — adds
   fee_category column using cut or apply
4. summarise(df, group_col, agg_col) — groups
   by group_col and returns sum and mean
   of agg_col
5. export_csv(df, filepath) — saves without index

Then create main.py that:
- Imports from data_utils
- Calls each function in sequence on
  appointments.csv
- Prints summary and exports result

This simulates how real Python pipeline
code is organized in production.

Concepts: module, import, def, clean separation
          of concerns, reusable functions'''

In [6]:
'''Exercise 5 - Pandas: Date operations

Load appointments_clean.csv from Day 9.
The appt_date column should already be datetime.
If not, convert it using pd.to_datetime().

1. Extract year, month, day into
   separate columns.
2. Extract day of week name.
3. Find which month had the most appointments.
4. Calculate days elapsed since each appointment
   to today using pd.Timestamp.today().
5. Filter appointments that happened
   in the first 10 days of January 2024.
6. Sort appointments by date ascending
   and reset the index.

Concepts: dt.year, dt.month, dt.day,
          dt.day_name(), pd.Timestamp.today(),
          timedelta arithmetic, sort_values()
'''

import pandas as pd

# 1. Load CSV
df = pd.read_csv("appointments_clean.csv")

# Ensure appt_date is datetime
df["appt_date"] = pd.to_datetime(df["appt_date"], errors="coerce")

# 2. Extract year, month, day
df["year"] = df["appt_date"].dt.year
df["month"] = df["appt_date"].dt.month
df["day"] = df["appt_date"].dt.day

# 3. Extract day of week name
df["day_name"] = df["appt_date"].dt.day_name()

# 4. Find which month had the most appointments
month_counts = df["month"].value_counts()
most_appts_month = month_counts.idxmax()
print("Month with most appointments:", most_appts_month)

# 5. Calculate days elapsed since each appointment
today = pd.Timestamp.today().normalize()
df["days_elapsed"] = (today - df["appt_date"]).dt.days

# 6. Filter appointments in first 10 days of Jan 2024
jan_filter = (df["year"] == 2024) & (df["month"] == 1) & (df["day"] <= 10)
jan_appts = df.loc[jan_filter]
print("Appointments in first 10 days of Jan 2024:")
print(jan_appts)

# 7. Sort appointments by date ascending and reset index
df_sorted = df.sort_values("appt_date").reset_index(drop=True)

# Final output
print("\nSorted DataFrame:")
print(df_sorted.head())


Month with most appointments: 1
Appointments in first 10 days of Jan 2024:
   appt_id  patient_id  doctor_id  appt_date             diagnosis  fee  \
0        1           1          1 2024-01-10  Hypertension checkup  500   

  fee_category day_of_week  year  month  day   day_name  days_elapsed  
0       Medium   Wednesday  2024      1   10  Wednesday           848  

Sorted DataFrame:
   appt_id  patient_id  doctor_id  appt_date             diagnosis  fee  \
0        1           1          1 2024-01-10  Hypertension checkup  500   
1        2           2          5 2024-01-11        Fever and cold  200   
2        3           3          3 2024-01-12  Knee pain evaluation  750   
3        4           4          2 2024-01-13    Migraine treatment  600   
4        5           5          4 2024-01-14        Acne treatment  300   

  fee_category day_of_week  year  month  day   day_name  days_elapsed  
0       Medium   Wednesday  2024      1   10  Wednesday           848  
1          Low  